# Deep Ensembles: train members on GPU (CISC 473 P23)
Trains ResNet-20 members with the exact midterm recipe (CIFAR-10 60/20/20 split seed 0, SGD-Nesterov lr 0.1, wd 5e-4, bs 128, cosine 40 epochs, crop+flip),
dumps softmax probabilities for each finished member, and zips this notebook's seeds into `de_gpu_seeds<first>-<last>.zip`.

**To split across two notebooks:** run one with `SEEDS = range(5, 10)` (ensemble B) and another with `SEEDS = range(10, 15)` (ensemble C).
Works on Python 3.10-3.12 with the preinstalled torch/numpy/scipy; nothing is pinned or reinstalled. See the instructions in the chat / README.

In [ ]:
# Cell 1: fail fast
import sys, torch
print("python", sys.version.split()[0], "| torch", torch.__version__, "| cuda build", torch.version.cuda, "| cuda available", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Kaggle Settings > Accelerator > GPU (T4/P100); Colab Runtime > Change runtime type > GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Cell 2: the only thing you edit
SEEDS = range(5, 10)        # notebook 1: range(5, 10)   notebook 2: range(10, 15)
REPO_URL = "https://github.com/spy772/CISC-473-P23.git"
BRANCH = "de-final"
SEEDS = list(SEEDS)
assert all(s >= 5 for s in SEEDS), "seeds 0-4 already exist on the laptop; do not retrain them"

In [ ]:
# Cell 3: clone the repo, install only what is missing (unpinned)
import os, subprocess, importlib
if not os.path.exists("CISC-473-P23"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, "CISC-473-P23"], check=True)
WORK = os.getcwd()
os.chdir("CISC-473-P23")
missing = [m for m in ("numpy", "scipy", "torchvision") if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, check=True)
print("missing packages installed:", missing or "none", "| cwd:", os.getcwd())
assert os.path.exists("data/cifar-10-batches-py/data_batch_1"), "CIFAR-10 batches should be in the repo"
# SVHN official test split (26,032 imgs) + hub model definition need internet: fail early
subprocess.run([sys.executable, "-c", "import sys; sys.path.insert(0,'src'); import dump_member_probs as d; print('SVHN test', d.load_svhn_test().shape)"], check=True)

In [ ]:
# Cell 4: train -> dump probs -> re-zip, one member at a time (a disconnect only loses the member in progress;
# re-running this cell skips finished members and resumes a partly trained one from its last epoch)
import glob, re, zipfile, json, time
ZIP = os.path.join(WORK, "de_gpu_seeds%d-%d.zip" % (min(SEEDS), max(SEEDS)))   # outside the repo -> shows up in Kaggle Output

def make_zip():
    files = []
    for s in SEEDS:
        files += glob.glob("results/raw/de_members/seed%d/*" % s)
        files += glob.glob("models/deep_ensembles/resnet20_seed%d.pt" % s)
        files += glob.glob("models/deep_ensembles/train_log_seed%d.csv" % s)
        files += glob.glob("models/deep_ensembles/train_time_seed%d.json" % s)
    with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        for f in files:
            z.write(f)
    print("  zip updated:", ZIP, len(files), "files,", round(os.path.getsize(ZIP) / 1e6, 1), "MB")

for s in SEEDS:
    t0 = time.time()
    subprocess.run([sys.executable, "models/deep_ensembles/train_ensemble.py", "--seeds", str(s)], check=True)
    subprocess.run([sys.executable, "src/dump_member_probs.py", "--seeds", str(s)], check=True)
    make_zip()
    print("seed", s, "finished in %.1f min" % ((time.time() - t0) / 60))

In [ ]:
# Cell 5: summary
print("seed  train_min  test_acc  hardware")
for s in SEEDS:
    t = json.load(open("models/deep_ensembles/train_time_seed%d.json" % s))
    m = json.load(open("results/raw/de_members/seed%d/meta.json" % s))
    print("%4d  %9.1f  %.4f    %s" % (s, t["wall_clock_sec"] / 60, m["test_acc"], m["hardware"]))
print("Download:", ZIP)
# Colab only: from google.colab import files; files.download(ZIP)